In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

pm25 = pd.read_csv(
    project_dir / "data/processed/pm25_cleaned.csv",
    dtype={
        "State Code": str,
        "County Code": str,
        "Site Num": str
    }
)

pm25["station_id"] = (
    pm25["State Code"].str.zfill(2)
    + "-"
    + pm25["County Code"].str.zfill(3)
    + "-"
    + pm25["Site Num"].str.zfill(4)
)

stations = (
    pm25[
        ["station_id", "Latitude", "Longitude"]
    ]
    .drop_duplicates("station_id")
    .reset_index(drop=True)
)

print("Total monitoring stations:", len(stations))

display(stations.head())


Total monitoring stations: 97


,station_id,Latitude,Longitude
0,06-007-0008,39.761680,-121.840470
1,06-009-0001,38.201850,-120.680277
2,06-011-0007,39.021221,-122.281803
3,06-011-1002,39.189190,-121.998870
4,06-019-0011,36.785380,-119.773210


In [2]:

import requests
import time

URL = "https://archive-api.open-meteo.com/v1/archive"

def fetch_station_weather(station):
    params = {
        "latitude": float(station["Latitude"]),
        "longitude": float(station["Longitude"]),
        "start_date": "2025-07-01",
        "end_date": "2025-09-30",
        "hourly": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m",
            "wind_direction_10m"
        ]),
        "timezone": "America/Los_Angeles"
    }

    response = requests.get(
        URL,
        params=params,
        timeout=90
    )
    response.raise_for_status()

    data = pd.DataFrame(response.json()["hourly"])

    data["station_id"] = station["station_id"]

    return data


In [3]:

weather_results = []

for _, station in stations.head(3).iterrows():
    print("Fetching:", station["station_id"])

    station_weather = fetch_station_weather(station)

    weather_results.append(station_weather)

    time.sleep(1)

weather_test = pd.concat(
    weather_results,
    ignore_index=True
)

print("Total records:", len(weather_test))
print("Stations:", weather_test["station_id"].nunique())

display(weather_test.head())


Fetching: 06-007-0008
Fetching: 06-009-0001
Fetching: 06-011-0007
Total records: 6624
Stations: 3


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,station_id
0,2025-07-01T00:00,29.7,29,0.0,13.6,101,06-007-0008
1,2025-07-01T01:00,28.9,30,0.0,14.0,126,06-007-0008
2,2025-07-01T02:00,27.7,32,0.0,10.8,116,06-007-0008
3,2025-07-01T03:00,26.7,34,0.0,6.2,128,06-007-0008
4,2025-07-01T04:00,25.8,37,0.0,6.4,148,06-007-0008


In [4]:

from pathlib import Path

weather_folder = (
    project_dir / "data/raw/weather/stations"
)

weather_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Weather cache folder ready:")
print(weather_folder)


Weather cache folder ready:
/Users/sumanthkamath/Documents/VS Code /EcoSentinel/data/raw/weather/stations


In [5]:

import time
import requests

successful = []
failed = []

for i, station in stations.iterrows():

    station_id = station["station_id"]

    file_path = (
        weather_folder / f"{station_id}_weather.csv"
    )

    # Skip previously downloaded stations
    if file_path.exists():
        print(f"Already downloaded: {station_id}")
        successful.append(station_id)
        continue

    try:
        print(
            f"Downloading {i + 1}/{len(stations)}: "
            f"{station_id}"
        )

        data = fetch_station_weather(station)

        # Check that the response contains expected hours
        if len(data) != 2208:
            raise ValueError(
                f"Unexpected record count: {len(data)}"
            )

        data.to_csv(file_path, index=False)

        successful.append(station_id)

        # Small delay between requests
        time.sleep(2)

    except (requests.RequestException, ValueError, KeyError) as e:
        print(f"Failed: {station_id} — {e}")
        failed.append(station_id)

print("\nDownload completed.")
print("Successful:", len(successful))
print("Failed:", len(failed))

if failed:
    print("Failed stations:", failed)



Download completed.
Successful: 97
Failed: 0


In [6]:

import pandas as pd

files = sorted(weather_folder.glob("*_weather.csv"))

print("Weather files found:", len(files))

# Ensure all stations are available
assert len(files) == len(stations), (
    "Some station weather files are missing."
)

weather_all = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

print("Total hourly records:", len(weather_all))
print("Unique stations:", weather_all["station_id"].nunique())

assert len(weather_all) == 214176
assert weather_all["station_id"].nunique() == 97

display(weather_all.head())


Weather files found: 97
Total hourly records: 214176
Unique stations: 97


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,station_id
0,2025-07-01T00:00,29.7,29,0.0,13.6,101,06-007-0008
1,2025-07-01T01:00,28.9,30,0.0,14.0,126,06-007-0008
2,2025-07-01T02:00,27.7,32,0.0,10.8,116,06-007-0008
3,2025-07-01T03:00,26.7,34,0.0,6.2,128,06-007-0008
4,2025-07-01T04:00,25.8,37,0.0,6.4,148,06-007-0008


In [7]:

import numpy as np

weather_all["time"] = pd.to_datetime(
    weather_all["time"]
)

weather_all["date"] = (
    weather_all["time"].dt.normalize()
)

# Wind direction: meteorological FROM direction
wind_rad = np.deg2rad(
    weather_all["wind_direction_10m"]
)

weather_all["wind_sin"] = np.sin(wind_rad)
weather_all["wind_cos"] = np.cos(wind_rad)

# Aggregate hourly data by station and date
weather_daily = (
    weather_all.groupby(
        ["station_id", "date"]
    )
    .agg(
        temperature_mean=("temperature_2m", "mean"),
        humidity_mean=("relative_humidity_2m", "mean"),
        wind_speed_mean=("wind_speed_10m", "mean"),
        precipitation_total=("precipitation", "sum"),
        wind_sin_mean=("wind_sin", "mean"),
        wind_cos_mean=("wind_cos", "mean"),
        hourly_records=("time", "count")
    )
    .reset_index()
)

# Circular mean wind direction
weather_daily["wind_direction_mean"] = (
    np.degrees(
        np.arctan2(
            weather_daily["wind_sin_mean"],
            weather_daily["wind_cos_mean"]
        )
    ) % 360
)

print("Daily weather records:", len(weather_daily))

print("\nHourly observations per station-day:")
print(weather_daily["hourly_records"].value_counts())

print("\nMissing values:")
print(weather_daily.isna().sum())

display(weather_daily.head())


Daily weather records: 8924

Hourly observations per station-day:
hourly_records
24    8924
Name: count, dtype: int64

Missing values:
station_id             0
date                   0
temperature_mean       0
humidity_mean          0
wind_speed_mean        0
precipitation_total    0
wind_sin_mean          0
wind_cos_mean          0
hourly_records         0
wind_direction_mean    0
dtype: int64


,station_id,date,temperature_mean,humidity_mean,wind_speed_mean,precipitation_total,wind_sin_mean,wind_cos_mean,hourly_records,wind_direction_mean
0,06-007-0008,2025-07-01,31.308333,28.083333,9.545833,0.0,0.139949,-0.240726,24,149.827864
1,06-007-0008,2025-07-02,31.233333,29.750000,7.625000,0.0,0.283526,-0.347032,24,140.751112
2,06-007-0008,2025-07-03,28.366667,35.916667,12.516667,0.0,0.082590,-0.668494,24,172.956986
3,06-007-0008,2025-07-04,25.512500,32.375000,15.720833,0.0,0.537723,-0.817414,24,146.661723
4,06-007-0008,2025-07-05,25.183333,28.666667,7.237500,0.0,-0.072326,-0.524710,24,187.848240


In [8]:

output_path = (
    project_dir /
    "data/processed/statewide_weather_daily.csv"
)

weather_daily.to_csv(
    output_path,
    index=False
)

print("Saved daily weather dataset:")
print(output_path)


Saved daily weather dataset:
/Users/sumanthkamath/Documents/VS Code /EcoSentinel/data/processed/statewide_weather_daily.csv
